In [1]:
import openeo
from openeo.processes import quantiles
from shapely.geometry import shape
import os
import geopandas as gpd
from scipy.ndimage import median_filter
import xarray as xr
import rasterio
import glob
from scipy.signal import find_peaks
from rasterio.mask import mask
from rasterio.merge import merge
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

In [ ]:
# Remove connection to the backend and remove the refresh token 
#from openeo.rest.auth.config import RefreshTokenStore
#RefreshTokenStore().remove()

In [2]:
connection = openeo.connect(url="openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

## Sentinel-2 mosaic from Sentinel-2 L2A

Create the Sentinel-2 mosaic from L2A data. Product description: 

https://dataspace.copernicus.eu/news/2024-2-27-exploring-new-frontier-sentinel-cloudless-mosaics-copernicus-data-space-ecosystem

https://documentation.dataspace.copernicus.eu/Data/SentinelMissions/Sentinel2.html#sentinel-2-level-3-quarterly-mosaics

In [3]:
spatial_extent_west = {"west": 610950, "east": 674650, "south": 5143820, "north": 5206380, "crs": "EPSG:32632"} # Western South Tyrol glaciers
spatial_extent_east = {"west": 704100, "east": 747040, "south": 5195880, "north": 5219660, "crs": "EPSG:32632"} # Eastern South Tyrol glaciers

In [ ]:
temporal_extent = ["2018-07-01T00:00:00Z", "2018-09-30T23:59:59Z"]

In [27]:
output_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests")
year = 2017

### 1) Generate the Sentinel-2 L2A cube

Load all the 10 m resolution bands + SCL

In [31]:
def load_s2(spatial_extent):
    s2_cube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
        bands=["B02", "B03", "B04", "B08","B11" ,"B12" ,"SCL"],
        max_cloud_cover=50
)
# Resample the whole cube to 10m so B11 aligns with B03/B04/B08
    return s2_cube.resample_spatial(resolution=10, method="bilinear")

### 2) Create the mask from SCL

Create the mask using the following SCL values:
```
1  = SC_SATURATED_DEFECTIVE
3  = SC_CLOUD_SHADOW
7  = SC_CLOUD_LOW_PROBA / UNCLASSIFIED
8  = SC_CLOUD_MEDIUM_PROBA
9  = SC_CLOUD_HIGH_PROBA
10 = SC_THIN_CIRRUS
```

In [20]:
def mask_clouds(cube):
    scl = cube.band("SCL")
    return (
    (scl == 1) |
    (scl == 3) |
    (scl == 7) |
    (scl == 8) |
    (scl == 9) |
    (scl == 10)
    )

### 3) Create S2 mosaic for east and west

In [33]:
def create_s2_mosaic(spatial_extent):

    cube = load_s2(spatial_extent)

    mask = mask_clouds(cube)

    observations = (
        mask.reduce_dimension("t", "sum")
        .add_dimension("bands", "observations", type="bands")
    )

    mosaic = cube.mask(mask).reduce_dimension(
        dimension="t",
        reducer=lambda x: quantiles(x, probabilities=[0.25])
    )

    return mosaic.merge_cubes(observations)

### 4) Send job and execute and wait

In [ ]:
import time

regions = {
    "west": spatial_extent_west,
    "east": spatial_extent_east,
}

jobs = {}

for name, extent in regions.items():

    print(f"Starting {name}...")

    cube = create_s2_mosaic(extent)

    job = cube.create_job(title=f"S2_mosaic_{name}_{year}")
    job.start()

    jobs[name] = job

    print(f"{name} started: {job.job_id}")

    time.sleep(30)
    

Starting west...
west started: j-26091513435440649d10840867386e70
Starting east...
east started: j-2609151344324d6fb3aa4f636dbd7f6a


### 5) Download both results

In [41]:
job_statuses = {name: job.status() for name, job in jobs.items()}

In [66]:
job_statuses

{'west': 'running', 'east': 'running'}

In [ ]:
job_results = {name: job.get_results() for name, job in jobs.items()}

# download the results to the output folder
for name, result in job_results.items():
    result.download_files(output_folder / name)

[PosixPath('/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/west/openEO.tif'),
 PosixPath('/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/west/job-results.json')]

### 6) Load both mosaics from disc

In [22]:
in_folder = output_folder / "s2_mosaics/"
in_folder.mkdir(parents=True, exist_ok=True)

print("Looking in:", in_folder)
print("Folder exists:", in_folder.exists())

mosaic_filenames = list()

for f in glob.glob(os.path.join(in_folder, 'S2_mosaic*.tiff')): 
    with rasterio.open(f) as src: 
        mosaic_filenames.append(f) 
        print(f"File: {f}, CRS: {src.crs}, Bounds: {src.bounds}, Width: {src.width}, Height: {src.height}")

Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics
Folder exists: True
File: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/S2_mosaic_west_2016.tiff, CRS: EPSG:32632, Bounds: BoundingBox(left=610950.0, bottom=5143820.0, right=674650.0, top=5206380.0), Width: 6370, Height: 6256
File: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/S2_mosaic_east_2023.tiff, CRS: EPSG:32632, Bounds: BoundingBox(left=704100.0, bottom=5195880.0, right=747040.0, top=5219660.0), Width: 4294, Height: 2378
File: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/S2_mosaic_east_2017.tiff, CRS: EPSG:32632, Bounds: BoundingBox(left=704100.0, bottom=5195880.0, right=747040.0, top=5219660.0), Width: 4294, Height: 2378
File: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/S2_mosaic_west_2023.tiff, CRS: EPSG:32632, Bounds: BoundingBox(left=610950.0, bottom=5143820.0, right=674650.0, top=5206380.0), Widt

### 7) Calculate ratios and ndsi

In [23]:
# calculate red/swir ratio and ndsi for each mosaic
out_ndsi = output_folder / "s2_mosaics" / "ndsi"
out_ndsi.mkdir(parents=True, exist_ok=True)

out_red_swir = output_folder / "s2_mosaics" / "ratio"
out_red_swir.mkdir(parents=True, exist_ok=True)

for f in mosaic_filenames:
    f = Path(f)
    with rasterio.open(f) as src:
        red = src.read(3).astype(np.float32)
        nir = src.read(4).astype(np.float32)
        swir = src.read(5).astype(np.float32)
        green = src.read(2).astype(np.float32)

        nodata = src.nodata
        valid = (swir > 0) & (red >= 0) & (green >= 0) & (nir >= 0)
        if nodata is not None:
            valid &= (red != nodata) & (swir != nodata) & (green != nodata) & (nir != nodata)

        # Red/SWIR ratio 
        red_swir_ratio = np.where(
            valid,
            red / np.where(swir == 0, np.nan, swir),
            np.nan
        )

        # nir/swir ratio
        nir_swir_ratio = np.where(
            valid,
            nir / np.where(swir == 0, np.nan, swir),
            np.nan
        )

        # NDSI 
        denom = green + swir
        ndsi = np.where(
            valid,
            (green - swir) / np.where(denom == 0, np.nan, denom),
            np.nan
        )

        # Save the results as new GeoTIFFs
        profile = src.profile
        profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)

        red_swir_filename = out_red_swir / f"{f.stem}_red_swir_ratio.tiff"
        ndsi_filename = out_ndsi / f"{f.stem}_ndsi.tiff"
        nir_swir_filename = out_red_swir / f"{f.stem}_nir_swir_ratio.tiff"

        with rasterio.open(red_swir_filename, 'w', **profile) as dst:
            dst.write(red_swir_ratio.astype(rasterio.float32), 1)

        with rasterio.open(ndsi_filename, 'w', **profile) as dst:
            dst.write(ndsi.astype(rasterio.float32), 1)

        with rasterio.open(nir_swir_filename, 'w', **profile) as dst:
            dst.write(nir_swir_ratio.astype(rasterio.float32), 1)

/tmp/ipykernel_7109/610697707.py:11: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  red = src.read(3).astype(np.float32)
/tmp/ipykernel_7109/610697707.py:12: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  nir = src.read(4).astype(np.float32)
/tmp/ipykernel_7109/610697707.py:13: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  swir = src.read(5).astype(np.float32)
/tmp/ipykernel_7109/610697707.py:14: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  green = src.

### 8) Based on thresholds create binary masks

Delineate glacier outline

In [24]:
# Create binary raster for each mosaic based on NDSI threshold
for f in mosaic_filenames:
    f = Path(f)
    ndsi_filename = out_ndsi / f"{f.stem}_ndsi.tiff"
    binary_filename = out_ndsi / f"{f.stem}_binary_ndsi_thresh_0.4.tiff"

    print("Foundn ndsi files:", ndsi_filename)

    with rasterio.open(ndsi_filename) as src:
        ndsi = src.read(1)
        binary_raster = (ndsi > 0.4)

        profile = src.profile
        #profile.update(dtype=rasterio.int32, count=1)

        # write to folder ndsi 
        with rasterio.open(binary_filename, 'w', **profile) as dst:
            dst.write(binary_raster, 1)

for f in mosaic_filenames:
    f = Path(f)
    red_swir_filename = out_red_swir / f"{f.stem}_red_swir_ratio.tiff"
    binary_filename = out_red_swir / f"{f.stem}_binary_red_swir_thresh_5.tiff"


    print("Foundn ratios files:", red_swir_filename)

    with rasterio.open(red_swir_filename) as src:
        red_swir_ratio = src.read(1)
        binary_raster = (red_swir_ratio > 5)

        profile = src.profile
        #profile.update(dtype=rasterio.int32, count=1)

        with rasterio.open(binary_filename, 'w', **profile) as dst:
            dst.write(binary_raster, 1)

Foundn ndsi files: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ndsi/S2_mosaic_west_2016_ndsi.tiff


/tmp/ipykernel_7109/3009321137.py:10: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  ndsi = src.read(1)


Foundn ndsi files: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ndsi/S2_mosaic_east_2023_ndsi.tiff
Foundn ndsi files: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ndsi/S2_mosaic_east_2017_ndsi.tiff
Foundn ndsi files: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ndsi/S2_mosaic_west_2023_ndsi.tiff
Foundn ratios files: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ratio/S2_mosaic_west_2016_red_swir_ratio.tiff


/tmp/ipykernel_7109/3009321137.py:29: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  red_swir_ratio = src.read(1)


Foundn ratios files: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ratio/S2_mosaic_east_2023_red_swir_ratio.tiff
Foundn ratios files: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ratio/S2_mosaic_east_2017_red_swir_ratio.tiff
Foundn ratios files: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ratio/S2_mosaic_west_2023_red_swir_ratio.tiff


### 10) Mask the binary layers with the glacier outlines from 1997

In [14]:
# mask the binary rasters with the glacier outlines shapefile
mask_filename = "/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/glacier_outline/glacier_inventory/SGIhom_1997_v2025.shp"


# Read glacier mask
glacier_mask = gpd.read_file(mask_filename)

for f in mosaic_filenames:
    f = Path(f)
    red_swir_filename = out_red_swir / f"{f.stem}_binary_red_swir_thresh_5.tiff"
    masked_filename = out_red_swir / f"{f.stem}_red_swir_ratio_masked_5.tiff"

    with rasterio.open(red_swir_filename) as src:

        # Make sure glacier mask is in same CRS as raster
        glacier_mask_reprojected = glacier_mask.to_crs(src.crs)

        # Mask raster
        masked_array, masked_transform = mask(
            src,
            glacier_mask_reprojected.geometry,
            crop=True,
            nodata=np.nan
        )

        filtered_raster = median_filter(masked_array, size=3)

        # Copy raster metadata
        masked_meta = src.meta.copy()

        # Update metadata for cropped raster
        masked_meta.update({
            "height": masked_array.shape[1],
            "width": masked_array.shape[2],
            "transform": masked_transform,
            "nodata": np.nan
        })

        # Save
        with rasterio.open(masked_filename, "w", **masked_meta) as dst:
            dst.write(filtered_raster)

        print("Saved files:", masked_filename)


Saved files: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ratio/S2_mosaic_west_2016_red_swir_ratio_masked_5.tiff
Saved files: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ratio/S2_mosaic_east_2023_red_swir_ratio_masked_5.tiff
Saved files: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ratio/S2_mosaic_east_2017_red_swir_ratio_masked_5.tiff
Saved files: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ratio/S2_mosaic_west_2023_red_swir_ratio_masked_5.tiff


In [25]:
# do the same for the NDSI rasters

for f in mosaic_filenames: 
    f = Path(f)
    ndsi_filename = out_ndsi / f"{f.stem}_binary_ndsi_thresh_0.4.tiff"
    masked_filename = out_ndsi / f"{f.stem}_binary_ndsi_masked_0.4.tiff"

    with rasterio.open(ndsi_filename) as src:

        # Make sure glacier mask is in same CRS as raster
        glacier_mask_reprojected = glacier_mask.to_crs(src.crs)

        # Mask raster
        masked_array, masked_transform = mask(
            src,
            glacier_mask_reprojected.geometry,
            crop=True,
            nodata=np.nan
        )

        filtered_raster = median_filter(masked_array, size=3)

        # Copy raster metadata
        masked_meta = src.meta.copy()

        # Update metadata for cropped raster
        masked_meta.update({
            "height": masked_array.shape[1],
            "width": masked_array.shape[2],
            "transform": masked_transform,
            "nodata": np.nan
        })

        # Save
        with rasterio.open(masked_filename, "w", **masked_meta) as dst:
            dst.write(filtered_raster)

        print("Saved files:", masked_filename)


Saved files: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ndsi/S2_mosaic_west_2016_binary_ndsi_masked_0.4.tiff
Saved files: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ndsi/S2_mosaic_east_2023_binary_ndsi_masked_0.4.tiff
Saved files: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ndsi/S2_mosaic_east_2017_binary_ndsi_masked_0.4.tiff
Saved files: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ndsi/S2_mosaic_west_2023_binary_ndsi_masked_0.4.tiff


### 11) Create a composite of west and east ndsi and ratio

In [28]:
in_folder = output_folder / "s2_mosaics/ndsi"

year = 2017

print("Looking in:", in_folder)

mosaic_filenames = list()

# change pattern accordingly, e.g. year 2023, 2016 etc.
for f in glob.glob(os.path.join(in_folder, f'S2_mosaic_*_{year}_binary_ndsi_masked_0.4.tiff')):
    with rasterio.open(f) as src:
        mosaic_filenames.append(f)
        print(f"File: {f}, CRS: {src.crs}, Bounds: {src.bounds}, Width: {src.width}, Height: {src.height}")

# Merge
mosaic, out_transform = merge(mosaic_filenames, method='max')

# Copy metadata from one source and update dimensions/transform
with rasterio.open(mosaic_filenames[0]) as ref:
    out_meta = ref.meta.copy()

out_meta.update({
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform,
    "dtype": "uint8",
    "nodata": 0,
})

with rasterio.open(in_folder / f"ndsi_composite_{year}_0.4.tiff", "w", **out_meta) as dest:
    dest.write(mosaic)


# do the same for the red/swir ratio rasters
in_folder_ratio = output_folder / "s2_mosaics/ratio"
print("Looking in:", in_folder_ratio)

mosaic_filenames = list()

for f in glob.glob(os.path.join(in_folder_ratio, f'S2_mosaic_*_{year}_red_swir_ratio_masked_5.tiff')):
    with rasterio.open(f) as src:
        mosaic_filenames.append(f)
        print(f"File: {f}, CRS: {src.crs}, Bounds: {src.bounds}, Width: {src.width}, Height: {src.height}")

# Merge
mosaic, out_transform = merge(mosaic_filenames, method='max')

# Copy metadata from one source and update dimensions/transform
with rasterio.open(mosaic_filenames[0]) as ref:
    out_meta = ref.meta.copy()

out_meta.update({
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform,
    "dtype": "uint8",
    "nodata": 0,
})

with rasterio.open(in_folder_ratio / f"ratio_composite_{year}.tiff", "w", **out_meta) as dest:
    dest.write(mosaic)

Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ndsi
File: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ndsi/S2_mosaic_east_2017_binary_ndsi_masked_0.4.tiff, CRS: EPSG:32632, Bounds: BoundingBox(left=704100.0, bottom=5195880.0, right=746060.0, top=5218920.0), Width: 4196, Height: 2304
File: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ndsi/S2_mosaic_west_2017_binary_ndsi_masked_0.4.tiff, CRS: EPSG:32632, Bounds: BoundingBox(left=610950.0, bottom=5144700.0, right=674650.0, top=5206380.0), Width: 6370, Height: 6168
Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ratio
File: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/ratio/S2_mosaic_west_2017_red_swir_ratio_masked_5.tiff, CRS: EPSG:32632, Bounds: BoundingBox(left=610950.0, bottom=5144700.0, right=674650.0, top=5206380.0), Width: 6370, Height: 6168
File: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/Glacie